## Imports

In [1]:
import pandas as pd
from pathlib import Path

## Variables globales

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "raw"

In [ ]:
dataFrames = {}
for csv_path in Path(DATA_DIR).glob("*.csv"):
    dataFrames[csv_path.stem] = pd.read_csv(csv_path)


## Description:

Dans cette section, nous présentons une description des différentes tables de notre jeu de données brutes.

### Forme:

In [ ]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.shape)

Les fichiers travel_times sont de loin les plus lourds, possédant environ 10 millions d'entrées chacuns.

In [ ]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.dtypes)


On observe deux clés utilisées dans notre jeu de données: node_id et link_id. Un champ grid_region_id est également présent mais n'apparait que dans nodes comme clé étrangère.

### échantillons de valeurs:

In [ ]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.head())

On peut observer que dans ce dataset, les booléens prennent la forme d'un string prenant les valeurs t/f. 
#### Links
Le champ osm_class semble être une énumération de valeurs. 
#### Nodes 
On confirme également que grid_region_id prends bien la forme d'un identifiant bien qu'il ne soit pour l'instant pas évident de déterminer ce qu'il référence, peut-être simplement un découpage de l'espace ?
Le champ osm_traffic_controller prends des valeurs nulles dans tout l'échantillon.

## Statistiques des valeurs:

### Valeurs nulles:

In [ ]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.isnull().sum())

Le jeu de données possède des valeurs nulles dans seulement deux champs:
- links.osm_name qui correspondent aux routes sans nom.
- nodes.osm_traffic_controller qui est nul pour toutes les entrées de la table. C'est donc un champ inutile que l'on pourra supprimer.

Au delà de ces deux champs, nos données n'ont aucune valeures manquantes ou incomplètes.

### Valeurs uniques:

In [ ]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.nunique())

#### links:
- osm_class possède 9 valeurs, ce qui comfirme qu'il s'agit bien d'une énumération
Ces valeurs sont:

In [ ]:
print(dataFrames["links"]["osm_class"].unique())


- birth et death_timestamp possèdent une seule valeur sur toute la table. Ils ne donnent donc aucune information sur les entrées et sont redondants

#### nodes:
- is_complete ne possède qu'une seule valeur: true, ce qui indique que toutes les entrées de nodes sont complètes, mais rend le champ redondant.
- grid_region_id possède un petit nombre de valeurs, il référence donc soit une petite table, soit un découpage géographique comme supposé plus tôt.

## Relations inter-tables: clé node_id

### Clés orphelines:

In [ ]:
primary_node_ids = set(dataFrames["nodes"]["node_id"].unique())

for df_name, df in dataFrames.items():
    if df_name == "nodes":
        continue
    
    print("------------- clés orphelines dans: " + df_name + " -------------")

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id:")
    print(len(df_begin_node_ids - primary_node_ids))

    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id:")
    print(len(df_end_node_ids - primary_node_ids))

#### links:
La table links contient un nombre non négligeable de clés orphelines, il faudra donc décider de garder ou non ces entrées en fonction de si elles peuvent nuire à la complétude du graphe.

#### travel_times:
Pour les tables travel_times, une seule valeur orpheline est rescencée:

In [ ]:
for df_name, df in dataFrames.items():
    if df_name == "nodes" or df_name == "links":
        continue

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id orpheline de " + df_name)
    print(df_begin_node_ids - primary_node_ids)


    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id orpheline de " + df_name)
    print(df_end_node_ids - primary_node_ids)


Toutes ces valeurs orphelines de node_id correspondent donc à l'identifiant 0. C'est un identifiant trop particulier pour que ce soit un simple oubli.
En effet, en regardant de plus près les trajets de travel_times qui partent de la node_id 0, on observe:

In [ ]:
print(dataFrames["travel_times_2011"][dataFrames["travel_times_2011"]["begin_node_id"] == 0]["end_node_id"].unique())
print(dataFrames["travel_times_2012"][dataFrames["travel_times_2012"]["begin_node_id"] == 0]["end_node_id"].unique())
print(dataFrames["travel_times_2013"][dataFrames["travel_times_2013"]["begin_node_id"] == 0]["end_node_id"].unique())
print(dataFrames["travel_times_2010"][dataFrames["travel_times_2010"]["begin_node_id"] == 0]["end_node_id"].unique())


Et pour ceux qui arrivent à la node_id 0:

In [ ]:
print(dataFrames["travel_times_2011"][dataFrames["travel_times_2011"]["end_node_id"] == 0]["begin_node_id"].unique())
print(dataFrames["travel_times_2012"][dataFrames["travel_times_2012"]["end_node_id"] == 0]["begin_node_id"].unique())
print(dataFrames["travel_times_2013"][dataFrames["travel_times_2013"]["end_node_id"] == 0]["begin_node_id"].unique())
print(dataFrames["travel_times_2010"][dataFrames["travel_times_2010"]["end_node_id"] == 0]["begin_node_id"].unique())

Cette valeur de node_id "orpheline" semble donc en réalité être une valeur par défaut attribuée aux trajets qui se passent sur des routes indéterminées, puisque tous les trajets contenant cette clé orpheline, se déplacent entre la node 0 et 0.

### Clées manquantes:

In [ ]:
for df_name, df in dataFrames.items():
    if df_name == "nodes":
        continue
    
    print("------------- clés manquantes dans: " + df_name + " -------------")

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id:")
    print(len(primary_node_ids - df_begin_node_ids))

    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id:")
    print(len(primary_node_ids - df_end_node_ids))

#### Dans links:

Dans la table links, aucune clé manquante n'est recensée, tous les sommets de graphe sont donc reliés.

#### Dans travel_times:
Les clés manquantes elles, sont moins problèmatiques, puisqu'elles révèlent juste la non complétude de certaines informations.
Il faut tout de même souligner que ces valeurs manquantes dans travel_times indique que nous n'avons pas de données sur le traffic de bon nombre de routes de la ville.